# Session 2 — Web Scraping with BeautifulSoup

**Data Science & Machine Learning: from Skills to the Project**  
**Duration:** 90 minutes  
**Case study:** extend Session 1's book catalogue using a small, locally generated HTML page.

The examples run offline and do not depend on an external website. No commercial sites, credentials or access restrictions are involved.

## Outcomes and agenda

By the end of this session, you can:
- distinguish a structured API from scraping rendered HTML and choose appropriately;
- inspect HTML tags, attributes and nesting, then select elements with CSS selectors;
- extract consistent records while handling missing elements;
- use focused regular expressions to clean prices, identifiers and text;
- build and validate a small, repeatable scraping pipeline.

| Minutes | Activity |
|---:|---|
| 0–10 | API or scraping? |
| 10–25 | HTML structure and selectors |
| 25–45 | BeautifulSoup and extraction |
| 45–55 | Regular expressions |
| 55–65 | Reliability and responsible access |
| 65–70 | Scrapy overview |
| 70–90 | Guided parser and independent challenge |

Run the notebook from top to bottom. The independent challenge is designed to take 20 minutes.

## 1. When should you scrape rather than use an API?

An API is usually the better choice when it provides the fields you need in a documented, structured format. Scraping can be appropriate when the information is publicly accessible in HTML and no suitable API or download is available. HTML is designed primarily for presentation: page layouts can change, fields may be absent, and a parser must account for variation.

Check the purpose, terms, permission, data licence, freshness and maintenance cost before choosing. Do not treat public visibility as permission to collect or reuse data.

**Interpretation:** for our lesson's small, stable product page, scraping practises HTML parsing. A documented catalogue API would be preferable for a real project if it exposes the same data.

In [ ]:
collection_options = {
    "documented API": {"format": "structured", "maintenance": "usually lower"},
    "HTML scraping": {"format": "presentation markup", "maintenance": "layout-dependent"},
}
assert collection_options["documented API"]["format"] == "structured"
assert collection_options["HTML scraping"]["maintenance"] == "layout-dependent"
print("For this exercise: use local HTML to practise extraction; prefer an API when suitable.")

## 2. HTML structure: tags, attributes and selectors

HTML is a tree. **Tags** describe elements (`article`, `h2`, `span`); **attributes** add information (`class`, `href`, `data-product-id`); child elements nest within parents. Inspect a page's structure before writing selectors. In browser developer tools, inspect the relevant element and check that the selector matches only the records you intend.

CSS selector examples: `article.product-card` selects articles with that class; `.price` selects elements by class; `[data-product-id]` selects elements with an attribute; `.product-card h2 a` selects links nested under product headings.

The generated fixture deliberately includes nested title markup, extra whitespace, a missing category, a missing price and ratings with unexpected wording.

In [ ]:
from pathlib import Path
import csv
import re
from urllib.parse import urljoin

import pandas as pd
import requests
from bs4 import BeautifulSoup

DATA_DIR = Path("data") / "raw"
DATA_DIR.mkdir(parents=True, exist_ok=True)
CATALOGUE_HTML = DATA_DIR / "catalogue_page.html"
SCRAPED_CSV = DATA_DIR / "catalogue_scraped.csv"

catalogue_html = """<!doctype html>
<html><head><title>Local Book Catalogue</title></head><body>
  <main id="catalogue">
    <article class="product-card" data-product-id="SKU-201" data-category="Fantasy">
      <h2><a href="/books/201">The Hobbit</a></h2>
      <p class="price">£8.99</p><span class="rating">4.5/5 stars</span>
    </article>
    <article class="product-card" data-product-id="SKU-202" data-category="Fiction">
      <h2><a href="/books/202">Pride and <em>Prejudice</em></a></h2>
      <p class="price">  £ 7.50 each </p><span class="rating">Rated 4/5</span>
    </article>
    <article class="product-card" data-product-id="SKU-203" data-category="Science">
      <h2><a href="/books/203">A Brief History of Time</a></h2>
      <p class="price">£9.25 approx.</p>
    </article>
    <article class="product-card" data-product-id="SKU-204">
      <h2><a href="/books/204">The Left Hand of Darkness</a></h2>
      <p class="price">£8.25</p><span class="rating">five stars</span>
    </article>
    <article class="product-card" data-product-id="SKU-205" data-category="Travel">
      <h2><a href="/books/205">Around the World in Eighty Days</a></h2>
      <span class="rating">3.5/5</span>
    </article>
  </main>
</body></html>"""
CATALOGUE_HTML.write_text(catalogue_html, encoding="utf-8")
assert CATALOGUE_HTML.exists()
print(f"Generated local HTML fixture: {CATALOGUE_HTML} ({catalogue_html.count('<article')} product cards)")

## 3. BeautifulSoup: `find`, `find_all` and `select`

`find()` returns the first matching element or `None`; `find_all()` returns all matches; `select()` and `select_one()` use CSS selectors. Check for a missing element before reading `.get_text()` or an attribute. A missing selector should be an explicit data-quality outcome, not an uncaught `AttributeError`.

**Interpretation:** the three lookup styles work on the same parsed document. SKU-205's missing price selector returns `None`; the title selector still matches all five links.

In [ ]:
soup = BeautifulSoup(CATALOGUE_HTML.read_text(encoding="utf-8"), "html.parser")
first_card = soup.find("article", class_="product-card")
all_cards = soup.find_all("article", class_="product-card")
title_links = soup.select(".product-card h2 a")
missing_price = soup.select_one('[data-product-id="SKU-205"] .price')

assert first_card is not None and first_card["data-product-id"] == "SKU-201"
assert len(all_cards) == len(title_links) == 5
assert missing_price is None
print("First card:", first_card.name, first_card.get("data-category"))
print("Cards:", len(all_cards), "| title links:", len(title_links), "| SKU-205 price element:", missing_price)

## 4. Extract records, handle gaps and clean text with regex

First define a consistent output schema. Treat `record_id` and `title` as required; preserve missing optional values as `None` rather than inventing data. Regex is useful for small, predictable text patterns: `re.sub(r"\s+", " ", text)` collapses whitespace, and `re.search()` can find a price or rating within a longer string. Regex does not replace HTML parsing—use BeautifulSoup to locate elements first.

**Interpretation:** `£ 7.50 each` becomes `7.5`; a written-out rating such as `five stars` remains missing for review rather than being guessed.

In [ ]:
def clean_text(value):
    """Collapse HTML text whitespace and return None for empty content."""
    if value is None:
        return None
    cleaned = re.sub(r"\s+", " ", value).strip()
    return cleaned or None

def parse_price(text):
    """Return a GBP amount found in text, or None when no price is present."""
    match = re.search(r"£\s*(\d+(?:\.\d{1,2})?)", text or "")
    return float(match.group(1)) if match else None

def parse_rating(text):
    """Return a numeric rating out of five, or None if it is not stated numerically."""
    match = re.search(r"\b(\d+(?:\.\d+)?)\s*/\s*5\b", text or "")
    if match is None:
        match = re.search(r"\bRated\s+(\d+(?:\.\d+)?)\b", text or "", re.IGNORECASE)
    return float(match.group(1)) if match else None

def parse_product_id(value):
    """Extract a three-digit SKU suffix; raise an informative error if invalid."""
    match = re.fullmatch(r"SKU-(\d{3})", value or "")
    if match is None:
        raise ValueError(f"Expected product ID in 'SKU-123' format; got {value!r}")
    return f"SKU-{match.group(1)}"

assert clean_text("  A Brief\n  History ") == "A Brief History"
assert parse_price("  £ 7.50 each ") == 7.5
assert parse_rating("4.5/5 stars") == 4.5
assert parse_rating("five stars") is None
assert parse_product_id("SKU-201") == "SKU-201"
print("Regex checks passed: price=7.5, rating=4.5, invalid numeric rating=None")

## 5. Guided lab — implement a reusable catalogue parser

The parser below takes an HTML file and its page URL, then returns one row per card. Required ID/title failures raise `ValueError` with the card number; absent or unparseable optional category, price and rating fields are `None`. The detail link is resolved against the page URL and retained as provenance. Read it, run it, then adapt a selector or quality rule if needed.

**Expected result:** five unique rows. SKU-203 has no rating, SKU-204 has no category and no numeric rating, and SKU-205 has no price. Missing optional values remain blank/`None`; nested title markup is cleaned. Required fields and duplicate IDs are checked.

In [ ]:
RECORD_COLUMNS = ["record_id", "title", "category", "price_gbp", "rating", "source_url"]

class CatalogueParseError(ValueError):
    """Raised when catalogue HTML cannot provide the required record structure."""

def parse_catalogue(html_path, page_url):
    """Parse local catalogue HTML into validated, consistently shaped records."""
    html = Path(html_path).read_text(encoding="utf-8")
    page = BeautifulSoup(html, "html.parser")
    cards = page.select("article.product-card")
    if not cards:
        raise CatalogueParseError(f"No product cards found in {html_path!s}; expected 'article.product-card'")

    records = []
    for number, card in enumerate(cards, start=1):
        raw_id = card.get("data-product-id")
        title_element = card.select_one("h2 a")
        title = clean_text(title_element.get_text(" ", strip=True)) if title_element else None
        if not raw_id or not title:
            raise CatalogueParseError(
                f"Product card {number} is missing required product ID or title"
            )

        product_id = parse_product_id(raw_id)
        price_element = card.select_one(".price")
        rating_element = card.select_one(".rating")
        link = title_element.get("href")
        records.append({
            "record_id": product_id,
            "title": title,
            "category": clean_text(card.get("data-category")),
            "price_gbp": parse_price(price_element.get_text(" ", strip=True)) if price_element else None,
            "rating": parse_rating(rating_element.get_text(" ", strip=True)) if rating_element else None,
            "source_url": urljoin(page_url, link) if link else page_url,
        })

    frame = pd.DataFrame.from_records(records, columns=RECORD_COLUMNS)
    if frame["record_id"].duplicated().any():
        duplicates = frame.loc[frame["record_id"].duplicated(), "record_id"].tolist()
        raise CatalogueParseError(f"Duplicate product IDs found: {duplicates}")
    return frame

page_url = "https://local.example/catalogue/"
scraped_df = parse_catalogue(CATALOGUE_HTML, page_url)
assert scraped_df.shape == (5, 6)
assert scraped_df["record_id"].is_unique
assert scraped_df[["record_id", "title"]].notna().all().all()
assert scraped_df["price_gbp"].isna().sum() == 1
assert scraped_df["category"].isna().sum() == 1
assert scraped_df["rating"].isna().sum() == 2
print(scraped_df[["record_id", "title", "category", "price_gbp", "rating"]].to_string(index=False))
print("Parsed records:", len(scraped_df), "| missing prices:", int(scraped_df["price_gbp"].isna().sum()))

## 6. Reliability and responsible access

Before making a request, read the site's terms and robots.txt policy and confirm that collection and reuse are permitted. Robots.txt is guidance for automated clients, not a substitute for permission or terms. Collect only what you need; avoid personal or sensitive data. Identify your client appropriately, keep request volumes low, respect published limits, and back off after `429` or transient server errors. Set timeouts and surface failures; do not retry indefinitely. Page structure can change, so validate expected selectors and values and review changes rather than silently exporting bad data.

This notebook parses local files and makes no requests to third-party sites by default. The optional cell only fetches the reserved example domain's robots.txt as a network demonstration; it does not crawl a commercial site.

In [ ]:
# Optional network demonstration: fetch only example.com's robots.txt, not page content.
RUN_LIVE_DEMO = False
if RUN_LIVE_DEMO:
    response = requests.get(
        "https://example.com/robots.txt",
        headers={"User-Agent": "DataScienceLearningNotebook/1.0"},
        timeout=5,
    )
    response.raise_for_status()
    print(f"HTTP {response.status_code}; robots.txt preview:")
    print("\n".join(response.text.splitlines()[:5]))
else:
    print("Network demonstration skipped; local HTML is sufficient for this session.")

## 7. Brief overview: when is Scrapy justified?

BeautifulSoup is a useful parser for a small number of already-fetched pages. **Scrapy** is a separate crawling framework worth considering when a project needs to follow many permitted links, manage crawl queues and concurrency, apply pipelines, or run a larger collection job. It is not needed for a single local page and is intentionally not installed or configured here.

**Interpretation:** start with the simplest tool that meets the permitted scope. A crawler increases the scale and responsibility of collection; it does not remove the need to respect site policies.

## 8. Save and validate the reusable dataset

The output follows a compact schema aligned with Session 1: `record_id` (unique SKU string), `title` (clean string), `category` (string or missing), `price_gbp` (non-negative number or missing), `rating` (numeric score out of five or missing), and `source_url` (detail-page URL or source page URL). Write missing values as empty CSV fields; retain missingness rather than filling it with a guessed value. The read-back assertions below confirm five rows, six columns, unique IDs and one blank price.

In [ ]:
def validate_catalogue(frame):
    """Raise an informative error when required catalogue quality rules fail."""
    missing_columns = set(RECORD_COLUMNS).difference(frame.columns)
    if missing_columns:
        raise ValueError(f"Catalogue is missing columns: {sorted(missing_columns)}")
    if frame.empty:
        raise ValueError("Catalogue contains no records")
    if frame["record_id"].isna().any() or not frame["record_id"].is_unique:
        raise ValueError("record_id values must be present and unique")
    if frame["title"].isna().any() or frame["title"].str.strip().eq("").any():
        raise ValueError("Every record must have a non-empty title")
    if (frame["price_gbp"].dropna() < 0).any():
        raise ValueError("Non-missing prices must be non-negative")
    if not frame["rating"].dropna().between(0, 5).all():
        raise ValueError("Non-missing ratings must be between 0 and 5")
    return True

assert validate_catalogue(scraped_df)
scraped_df.to_csv(SCRAPED_CSV, index=False, encoding="utf-8", na_rep="")
round_trip = pd.read_csv(SCRAPED_CSV)
assert round_trip.shape == (5, 6)
assert round_trip["record_id"].is_unique
assert round_trip["price_gbp"].isna().sum() == 1
print(f"Saved and re-read {len(round_trip)} validated records: {SCRAPED_CSV}")
print("Schema:", ", ".join(RECORD_COLUMNS))

## 9. Independent 20-minute challenge — second local page

Create a second local HTML page with **at least three** product cards. Reuse `parse_catalogue()` and `validate_catalogue()` to extract and validate it. Include one optional missing value and one price with extra surrounding text. Check that your parser returns the expected row count, unique IDs, non-negative prices and ratings in the range 0–5. If you choose to extend the HTML structure or parser, explain the selector change.

**Expected result:** three unique records, the first price parsed as `12.0`, and the third record's absent price retained as missing. All present ratings fall within 0–5. The shared parser works because both pages follow the same card structure.

In [ ]:
challenge_html_path = DATA_DIR / "catalogue_challenge.html"
challenge_html = """<!doctype html><html><body><main>
  <article class="product-card" data-product-id="SKU-301" data-category="History">
    <h2><a href="/books/301">The Silk Roads</a></h2>
    <p class="price">£12.00 paperback</p><span class="rating">4.2/5</span>
  </article>
  <article class="product-card" data-product-id="SKU-302" data-category="Poetry">
    <h2><a href="/books/302">Selected Poems</a></h2>
    <p class="price">£6.75</p><span class="rating">Rated 3.5</span>
  </article>
  <article class="product-card" data-product-id="SKU-303">
    <h2><a href="/books/303">The Sea, The Sea</a></h2>
    <span class="rating">5/5</span>
  </article>
</main></body></html>"""
challenge_html_path.write_text(challenge_html, encoding="utf-8")

challenge_df = parse_catalogue(challenge_html_path, "https://local.example/challenge/")
assert validate_catalogue(challenge_df)
assert len(challenge_df) == 3
assert challenge_df["record_id"].is_unique
assert challenge_df["price_gbp"].isna().sum() == 1
assert challenge_df["rating"].dropna().between(0, 5).all()
assert challenge_df.loc[0, "price_gbp"] == 12.0
print(challenge_df[["record_id", "title", "price_gbp", "rating"]].to_string(index=False))
print("Challenge passed:", len(challenge_df), "valid unique records")

## 10. Reflection and recap

1. When would you choose an API over scraping, even if the HTML is easier to view?
2. What does `select_one()` return when a matching element is absent, and how should the parser respond?
3. Why use BeautifulSoup for HTML and regex only for the price/rating text?
4. Which fields in this catalogue are required, and why should optional missing values remain missing?
5. What should you check before making requests to a real website, and what changes when a one-page task grows into a crawler?

**Expected points:** prefer a documented API when it provides suitable structured data; a missing selector returns `None`, so guard it; HTML has nesting that a markup parser understands, whereas regex is suited to small text patterns; required identifiers/titles should fail clearly while unknown optional values should not be fabricated; check permission, terms, robots guidance, data use and rate limits, and use a framework only when the permitted crawl scale justifies it.

**Recap:** inspect HTML before selecting; parse records with BeautifulSoup; use focused regex for text values; handle missing data and schema changes explicitly; validate and save only permitted, necessary data. The reusable output is `data/raw/catalogue_scraped.csv`.